# KaKaRoTiOS — Dragon Ball Z: Kakarot

Pipeline isolado na branch `KaKaRoTiOS`, a partir de `SpkIOSzero`. Os seis links AkiraBox já estão separados: **BASE** (`010051C0134F8000`, `v0`) e **UPDATE** (`010051C0134F8800`, `v1048576`), três volumes por conjunto. O notebook valida título, versão e ordem `part1/part2/part3` antes de baixar. Cada conjunto é extraído separadamente pelo RARLAB UnRAR; o resultado precisa ser exatamente um NSP ou XCI íntegro por conjunto.

Os identificadores AkiraBox do início dos nomes são diferentes em cada URL. Por isso, os arquivos são renomeados localmente para um mesmo nome-base por conjunto, preservando a ordem dos volumes. Base, update, RomFS e AOC ficam separados no Drive; o workspace de build também fica separado.

**Os volumes RAR só são apagados depois que os dois pacotes reconstruídos foram enviados e verificados no Drive.** A última célula é um despacho explícito e não executa por padrão.

In [ ]:
# Links fornecidos: BASE e UPDATE permanecem em grupos independentes.
DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"

BASE_PART_URLS = [
    "https://us1.akirabox.com/uploads/users/Z9dzBxMlmk1b/co6MsivArQWQHPYzkde-Dragon%20Ball%20Z%20Kakarot%20%5B010051C0134F8000%5D%5Bv0%5D%5Btaodung.com%5D.part1.rar?access=1794793813948.dK4oswIODUcamVU-EIfcT2Ph1hrDkKhI3xjJT67YGSA",  # BASE part1
    "https://us1.akirabox.com/uploads/users/Z9dzBxMlmk1b/cox1bB45oCHhv5ZEgde-Dragon%20Ball%20Z%20Kakarot%20%5B010051C0134F8000%5D%5Bv0%5D%5Btaodung.com%5D.part2.rar?access=1794793792478.9XG67t5SOSql6UDR6-NXVmUz5uu9bdrbKsNsRC3l_1o",  # BASE part2
    "https://us1.akirabox.com/uploads/users/Z9dzBxMlmk1b/coR8vVK6fU5T7byoCde-Dragon%20Ball%20Z%20Kakarot%20%5B010051C0134F8000%5D%5Bv0%5D%5Btaodung.com%5D.part3.rar?access=1794793890892.y-d8GufZXlmYX2yt0_ojsajAM4oONFvnu1esK2CuWDs",  # BASE part3
]
UPDATE_PART_URLS = [
    "https://us1.akirabox.com/uploads/users/Z9dzBxMlmk1b/co3GLxjdOISSwUMDwde-Dragon%20Ball%20Z%20Kakarot%20%5B010051C0134F8800%5D%5Bv1048576%5D%5Btaodung.com%5D.part1.rar?access=1794793930968.yif9SAW50NvXL4DRzOaolvumW0CHIQ3WduLIwaVg7fE",  # UPDATE part1
    "https://us1.akirabox.com/uploads/users/Z9dzBxMlmk1b/coAJWYzHg3DnQjCvXde-Dragon%20Ball%20Z%20Kakarot%20%5B010051C0134F8800%5D%5Bv1048576%5D%5Btaodung.com%5D.part2.rar?access=1794793959412.lVHdPgLvFnk7CIqX4rq58pfXVqvjMlaqQe2TmM20IBw",  # UPDATE part2
    "https://us1.akirabox.com/uploads/users/Z9dzBxMlmk1b/com0MbXa3ife0Jeprde-Dragon%20Ball%20Z%20Kakarot%20%5B010051C0134F8800%5D%5Bv1048576%5D%5Btaodung.com%5D.part3.rar?access=1794793980368.xNjROdgEKvWXQRyIhRxIZfQ8ij8hNffrQkJJkXp3MT8",  # UPDATE part3
]

# Nomes locais lógicos: o mesmo prefixo é necessário para o UnRAR encadear as partes.
# O prefixo aleatório AkiraBox varia por URL; estes nomes normalizam apenas os arquivos locais.
BASE_PART_NAMES = [
    "Dragon Ball Z Kakarot [010051C0134F8000][v0][taodung.com].part1.rar",
    "Dragon Ball Z Kakarot [010051C0134F8000][v0][taodung.com].part2.rar",
    "Dragon Ball Z Kakarot [010051C0134F8000][v0][taodung.com].part3.rar",
]
UPDATE_PART_NAMES = [
    "Dragon Ball Z Kakarot [010051C0134F8800][v1048576][taodung.com].part1.rar",
    "Dragon Ball Z Kakarot [010051C0134F8800][v1048576][taodung.com].part2.rar",
    "Dragon Ball Z Kakarot [010051C0134F8800][v1048576][taodung.com].part3.rar",
]

# Mantido desligado: nenhum GitHub Actions será disparado nesta preparação.
DISPATCH_TO_GITHUB = False

In [ ]:
from pathlib import Path
import argparse, json, os, subprocess, sys
from google.colab import auth

auth.authenticate_user()
ROOT = Path("/content/kakarot-ios-repo")
if not ROOT.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "KaKaRoTiOS",
        "https://github.com/joao9vegeto-stack/KintTany.git", str(ROOT)
    ], check=True)
else:
    branch = subprocess.check_output(
        ["git", "-C", str(ROOT), "branch", "--show-current"], text=True
    ).strip()
    if branch != "KaKaRoTiOS":
        raise RuntimeError(f"Workspace existente está na branch {branch!r}; não vou trocar de branch.")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "KaKaRoTiOS"], check=True)
    subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "origin/KaKaRoTiOS"], check=True)

assert subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip() == "KaKaRoTiOS"
sys.path.insert(0, str(ROOT))
PY_FILES = [
    "colab/source_kakarot.py", "colab/prepare_kakarot.py", "colab/run_kakarot.py",
    "scripts/prepare_kakarot_actions.py", "scripts/build_kakarot_shared.py",
]
subprocess.run([sys.executable, "-m", "py_compile", *[str(ROOT / item) for item in PY_FILES]], check=True)
print("KaKaRoTiOS Python syntax: OK")
print("Repository:", ROOT)
print("Branch:", "KaKaRoTiOS")

In [ ]:
from colab.run_kakarot import prepare, recover
from urllib.parse import unquote, urlparse

def assert_rar_group(urls, expected_title_id, expected_version, label):
    if len(urls) != 3:
        raise ValueError(f"{label}: são exigidos exatamente três links.")
    for number, url in enumerate(urls, 1):
        path = unquote(urlparse(url).path).lower()
        if not path.endswith(f".part{number}.rar"):
            raise ValueError(f"{label}: link {number} não é o volume part{number}.rar.")
        if f"[{expected_title_id.lower()}]" not in path or f"[{expected_version.lower()}]" not in path:
            raise ValueError(f"{label}: ID/versão inesperado no link {number}; interrompendo para não misturar pacotes.")

if all(x.strip() for x in BASE_PART_URLS + UPDATE_PART_URLS):
    assert_rar_group(BASE_PART_URLS, "010051C0134F8000", "v0", "BASE")
    assert_rar_group(UPDATE_PART_URLS, "010051C0134F8800", "v1048576", "UPDATE")
    handoff = prepare(argparse.Namespace(
        drive_folder_id=DRIVE_FOLDER_ID,
        base_part_urls=[x.strip() for x in BASE_PART_URLS],
        update_part_urls=[x.strip() for x in UPDATE_PART_URLS],
        base_part_names=BASE_PART_NAMES,
        update_part_names=UPDATE_PART_NAMES,
    ))
else:
    print("Os seis links ainda não estão preenchidos. Tentando recuperar uma preparação anterior do Drive...")
    handoff = recover(argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID))
    if handoff is None:
        raise ValueError("Preencha os três BASE_PART_URLS e os três UPDATE_PART_URLS, na ordem volume 1, 2, 3.")

print("\nPreparação recuperada/concluída:", handoff["request_id"])
print("Drive:", handoff["drive_output_folder_url"])

In [ ]:
# Auditoria antes de qualquer despacho para GitHub Actions.
prepared = handoff["prepared"]
print(json.dumps({
    "project": handoff["project"],
    "branch": handoff["branch"],
    "title_id": handoff["title_id"],
    "package_version": handoff["package_version"],
    "base_drive": handoff["game_data"]["base_package"],
    "update_drive": handoff["game_data"]["update_package"],
    "rar_parts_deleted": handoff["source"]["rar_parts_deleted_after_upload"],
    "profile_type": prepared["profile_type"],
    "profile_source": prepared["profile_source"],
    "main_build_id": prepared["main_build_id"],
    "main_sha256": prepared["main_sha256"],
    "modules": prepared["modules"],
    "graphics_api": prepared["graphics_api"],
    "unreal_shader_sources": prepared["unreal_shader_sources"],
    "pak_audit": prepared["pak_audit"],
    "shader_chain": prepared["shader_chain"],
    "romfs_size": prepared["romfs_size"],
    "romfs_sha256": prepared["romfs_sha256"],
    "aoc_count": prepared["aoc_count"],
    "aoc_images": prepared["aoc_images"],
    "drive_workspace": handoff["work_archive"],
}, indent=2))

In [ ]:
# DESPACHO EXPLÍCITO — só execute depois de revisar o handoff acima.
if DISPATCH_TO_GITHUB:
    from colab.run_kakarot import dispatch
    dispatch(argparse.Namespace(handoff=Path("/content/kakarot-ios/handoff.json")))
else:
    print("Actions NÃO disparada. Para enviar, altere DISPATCH_TO_GITHUB = True e execute esta célula.")